# HuggingFace Embeddings 실습

OpenAI가 아닌 HuggingFace 생태계의 임베딩 모델들을 실습한다. HuggingFace의 추론 엔드포인트(Endpoint)를 원격으로 호출하는 방법, 모델을 직접 내 컴퓨터로 내려받아 로컬에서 돌리는 방법, 그리고 다국어 임베딩에 강한 `BAAI/bge-m3` 모델을 LangChain 없이 `FlagEmbedding` 라이브러리로 직접 다루면서 dense·sparse·multi-vector(ColBERT) 세 가지 임베딩 방식까지 다룬다. HuggingFace API 토큰(`HF_TOKEN`)을 `.env`에서 불러와 사용한다.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## 1. 환경 설정

`HF_HOME` 환경변수로 HuggingFace 모델 다운로드 캐시 경로를 이 프로젝트 안의 `./cache/` 폴더로 지정한다(기본값은 사용자 홈 디렉터리 아래 공용 캐시라서, 프로젝트별로 별도 관리하고 싶을 때 이렇게 지정해준다). `warnings.filterwarnings("ignore")`는 모델 로딩 시 뜨는 부수적인 경고를 숨긴다.

In [2]:
import os
import warnings

warnings.filterwarnings("ignore")

os.environ["HF_HOME"] = "./cache/"

## 2. 임베딩할 예시 문장 준비

한국어·영어가 섞인 5개 문장을 준비한다. 이후 이 문장들을 임베딩해서 서로 의미가 얼마나 가까운지 비교해볼 것이다.

In [3]:
texts = [
    "안녕, 만나서 반가워.",
    "LangChain simplifies the process of building applications with large language models",
    "랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다. ",
    "LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.",
    "Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.",
]

## 3. HuggingFace Endpoint로 임베딩(원격 호출)

`HuggingFaceEndpointEmbeddings`는 모델을 내 컴퓨터에 내려받지 않고, HuggingFace의 추론 서버에 API로 요청을 보내 임베딩 결과만 받아오는 방식이다. `intfloat/multilingual-e5-large-instruct`는 다국어(한국어 포함)를 잘 지원하는 임베딩 모델이다. `task="feature-extraction"`은 (문장 생성이 아니라) 벡터 추출 작업임을 지정하는 옵션이고, API 인증에는 `HF_TOKEN`을 사용한다.

In [9]:
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings

model_name = "intfloat/multilingual-e5-large-instruct"

hf_embeddings = HuggingFaceEndpointEmbeddings(
    model=model_name,
    task="feature-extraction",
    huggingfacehub_api_token=os.environ["HF_TOKEN"]
)

`%%time`으로 5개 문장을 임베딩하는 데 걸린 시간을 측정한다. 원격 API 호출이라 네트워크 왕복 시간이 포함된다(실제 측정: 약 5.86초).

In [10]:
%%time
embedded_documents = hf_embeddings.embed_documents(texts)

CPU times: total: 250 ms
Wall time: 5.86 s


사용한 모델 이름과 임베딩 벡터의 차원 수(1024)를 확인한다.

In [11]:
print("[HuggingFace Endpoint Embedding]")
print(f"Model: \t\t{model_name}")
print(f"Dimension: \t{len(embedded_documents[0])}")

[HuggingFace Endpoint Embedding]
Model: 		intfloat/multilingual-e5-large-instruct
Dimension: 	1024


## 4. 질의(query) 임베딩 및 유사도 기반 검색

검색 질의 문장을 임베딩한다.

In [12]:
embedded_query = hf_embeddings.embed_query("LangChain에 대해서 알려주세요.")

질의 벡터도 문서와 같은 1024차원인지 확인한다(같은 벡터 공간에 있어야 서로 비교할 수 있다).

In [13]:
len(embedded_query)

1024

`numpy`의 행렬 곱(`@`)으로 질의 벡터와 각 문서 벡터 사이의 내적(유사도 점수 역할)을 한 번에 계산한다. `HuggingFaceEndpointEmbeddings`가 반환하는 벡터는 이미 정규화되어 있어서, 내적값을 코사인 유사도처럼 유사도 점수로 바로 사용할 수 있다.

In [14]:
import numpy as np

np.array(embedded_query) @ np.array(embedded_documents).T

array([0.84281223, 0.86560872, 0.86114495, 0.8970382 , 0.77247184])

유사도 점수를 내림차순으로 정렬한 인덱스를 구한다(`argsort()`는 오름차순 인덱스를 반환하므로 `[::-1]`로 뒤집어서 유사도가 높은 순서로 만든다).

In [15]:
sorted_idx = (np.array(embedded_query) @ np.array(embedded_documents).T).argsort()[::-1]
sorted_idx

array([3, 1, 2, 0, 4])

정렬된 순서대로 원문을 출력해서, 질의("LangChain에 대해서 알려주세요.")와 가장 관련 있는 문장이 위로 오는지 확인한다. LangChain을 직접 언급하는 한국어/영어 문장들이 상위에 오고, 관련 없는 인사말·RAG 문장은 하위로 밀린 것을 볼 수 있다.

In [16]:
print("[Query] LangChain에 대해서 알려주세요.\n========================================")
for i, idx in enumerate(sorted_idx):
    print(f"[{i}] {texts[idx]}")
    print()

[Query] LangChain에 대해서 알려주세요.
[0] LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.

[1] LangChain simplifies the process of building applications with large language models

[2] 랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다. 

[3] 안녕, 만나서 반가워.

[4] Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.



## 5. HuggingFaceEmbeddings로 로컬 실행

이번에는 모델을 원격 API가 아니라 내 컴퓨터로 직접 내려받아 실행하는 `HuggingFaceEmbeddings`를 사용한다. `model_kwargs={"device": "cpu"}`는 CPU에서 모델을 돌리겠다는 설정이고, `encode_kwargs={"normalize_embeddings": True}`는 임베딩 벡터를 정규화(크기를 1로 통일)해서, 내적만으로도 코사인 유사도와 같은 값을 얻을 수 있게 한다. 처음 실행 시 모델 가중치를 다운로드하고 불러오는 과정이 진행된다.

In [20]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

model_name = "intfloat/multilingual-e5-large-instruct"

hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 862.93it/s]


로컬 모델로 5개 문장을 임베딩하는 데 걸린 시간을 측정한다. 앞선 원격 API 호출과 비교해볼 수 있다.

In [21]:
%%time
embedded_documents1 = hf_embeddings.embed_documents(texts)

CPU times: total: 28 s
Wall time: 2.84 s


모델 이름과 차원 수를 출력한다(이 셀은 방금 만든 `embedded_documents1`이 아니라 4번 구간에서 만든 `embedded_documents`를 그대로 다시 참조하고 있다).

In [22]:
print(f"Model: \t\t{model_name}")
print(f"Dimension: ]t{len(embedded_documents[0])}")

Model: 		intfloat/multilingual-e5-large-instruct
Dimension: ]t1024


## 6. 다른 로컬 모델로 교체: BAAI/bge-m3

이번에는 다국어 검색에 강한 것으로 널리 쓰이는 `BAAI/bge-m3` 모델로 바꿔서 같은 방식(`HuggingFaceEmbeddings`)으로 로컬 실행해본다.

In [23]:
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "BAAI/bge-m3"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}
hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 48759.38it/s]


여기서는 `%%time`이 아니라 `%time`(한 줄짜리 라인 매직)을 쓰고, 그 다음 줄에 실제 임베딩 코드를 따로 적었다. `%time`은 같은 줄에 있는 코드만 시간을 재기 때문에, 다음 줄의 `embedded_documents = ...`는 측정 대상에 포함되지 않아 측정 시간이 0으로 나온다(의도한 측정이라면 `%%time`처럼 셀 전체를 재는 셀 매직을 써야 한다).

In [25]:
%time
embedded_documents = hf_embeddings.embed_documents(texts)

CPU times: total: 0 ns
Wall time: 0 ns


bge-m3 모델의 이름과 차원 수(1024)를 확인한다.

In [26]:
print(f"Model: \t\t{model_name}")
print(f"Dimension: \t{len(embedded_documents[0])}")

Model: 		BAAI/bge-m3
Dimension: 	1024


bge-m3로도 질의·문서 임베딩을 새로 만들어서, 내적 기반 유사도 정렬을 다시 확인해본다. e5 모델(4번)과 결과 순서가 정확히 같지는 않을 수 있는데, 모델마다 학습 데이터와 임베딩 공간이 다르기 때문이다.

In [27]:
import numpy as np

embedded_query = hf_embeddings.embed_query("LangChain에 대해서 알려주세요.")
embedded_documents = hf_embeddings.embed_documents(texts)

np.array(embedded_query) @ np.array(embedded_documents).T

sorted_idx = (np.array(embedded_query) @ np.array(embedded_documents).T).argsort()[::-1]

print("[Query] LangChain에 대해서 알려주세요.\n=====================================")
for i, idx in enumerate(sorted_idx):
    print(f"[{i}] {texts[idx]}")
    print()

[Query] LangChain에 대해서 알려주세요.
[0] LangChain simplifies the process of building applications with large language models

[1] LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.

[2] 랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다. 

[3] 안녕, 만나서 반가워.

[4] Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.



## 7. FlagEmbedding으로 BGE-M3 직접 사용하기

`BAAI/bge-m3`는 LangChain 래퍼 없이도 원 제작사가 만든 `FlagEmbedding` 라이브러리로 직접 쓸 수 있다. 이렇게 하면 이 모델의 세 가지 임베딩 방식(dense·sparse·multi-vector)을 모두 활용할 수 있다. 먼저 기본적인 dense(밀집) 벡터를 뽑아본다. `use_fp16=True`는 16비트 부동소수점 연산으로 속도를 높이는 옵션이고, `batch_size`/`max_length`는 한 번에 처리할 문장 수와 최대 토큰 길이다.

In [29]:
from FlagEmbedding import BGEM3FlagModel

model_name = "BAAI/bge-m3"
bge_embeddings = BGEM3FlagModel(
    model_name, use_fp16=True
)

bge_embedded = bge_embeddings.encode(
    texts,
    batch_size=12,
    max_length=8192,
)["dense_vecs"]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1258.02it/s]


결과 shape을 보면 `(5, 1024)` — 문장 5개, 각각 1024차원 벡터로 확인된다.

In [30]:
bge_embedded.shape

(5, 1024)

모델 이름과 차원 수를 다시 출력해 확인한다.

In [31]:
print(f"Model: \t\t{model_name}")
print(f"Dimension: \t{len(embedded_documents[0])}")

Model: 		BAAI/bge-m3
Dimension: 	1024


## 8. dense 벡터를 명시적으로 요청하기

`BGEM3FlagModel`을 새로 만들고, 이번에는 `encode()`에 `return_dense=True`를 명시적으로 지정해서 dense 벡터를 받는다(앞서는 기본 동작으로 dense_vecs를 받았다면, 여기서는 옵션으로 명시한 것).

In [32]:
from FlagEmbedding import BGEM3FlagModel

bge_flagmodel = BGEM3FlagModel(
    "BAAI/bge-m3", use_fp16=True
)

bge_encoded = bge_flagmodel.encode(
    texts,
    return_dense=True
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1370.99it/s]


마찬가지로 `(5, 1024)` shape을 확인한다.

In [33]:
bge_encoded["dense_vecs"].shape

(5, 1024)

## 9. sparse(어휘 기반) 벡터

이번에는 `return_sparse=True`로 sparse(희소) 벡터를 요청한다. sparse 벡터는 dense 벡터처럼 의미를 압축한 것이 아니라, 문장에 실제로 등장하는 단어(토큰)별 가중치를 담고 있어서 전통적인 키워드 검색과 비슷한 방식으로 활용할 수 있다(`lexical_weights`로 접근).

In [34]:
bge_flagmodel = BGEM3FlagModel(
    "BAAI/bge-m3", use_fp16=True
)

bge_encoded = bge_flagmodel.encode(texts, return_sparse=True)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1322.05it/s]


`compute_lexical_matching_score()`로 두 문장의 sparse 벡터(어휘 가중치) 사이의 매칭 점수를 계산한다. 같은 문장끼리 비교하면(`texts[0]` vs `texts[0]`) 공유하는 단어가 많아 점수가 나오고(0.3016), 언어와 내용이 전혀 다른 문장끼리 비교하면(`texts[0]` vs `texts[1]`, 한국어 vs 영어) 겹치는 단어가 없어 점수가 0이 된다. dense 벡터의 "의미적 유사도"와 달리, sparse 점수는 실제 단어가 얼마나 겹치는지를 반영한다.

In [35]:
lexical_scores1 = bge_flagmodel.compute_lexical_matching_score(
    bge_encoded["lexical_weights"][0], bge_encoded["lexical_weights"][0]
)

lexical_scores2 = bge_flagmodel.compute_lexical_matching_score(
    bge_encoded["lexical_weights"][0], bge_encoded["lexical_weights"][1]
)

print(lexical_scores1)
print(lexical_scores2)

0.30156046
0


## 10. ColBERT 방식의 multi-vector 임베딩

마지막으로 `return_colbert_vecs=True`로 ColBERT 방식의 임베딩을 요청한다. 문장 전체를 벡터 하나로 압축하는 대신, 문장을 이루는 토큰마다 각각의 벡터를 가지는 방식(multi-vector)이라서 더 세밀한 매칭이 가능하다.

In [36]:
bge_flagmodel = BGEM3FlagModel(
    "BAAI/bge-m3", use_fp16=True
)

bge_encoded = bge_flagmodel.encode(texts, return_colbert_vecs=True)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1629.95it/s]


`colbert_score()`로 두 문장의 ColBERT 벡터 사이의 유사도를 계산한다. 같은 문장끼리는 최대값인 1.0이 나오고, 다른 문장끼리는 0.3748로 dense/sparse와는 또 다른 관점의 유사도 점수가 나온다.

In [37]:
colbert_scores1 = bge_flagmodel.colbert_score(
    bge_encoded["colbert_vecs"][0], bge_encoded["colbert_vecs"][0]
)

colbert_scores2 = bge_flagmodel.colbert_score(
    bge_encoded["colbert_vecs"][0], bge_encoded["colbert_vecs"][1]
)

print(colbert_scores1)
print(colbert_scores2)

tensor(1.)
tensor(0.3748)


## 정리

| 방식 | 클래스/라이브러리 | 실행 위치 | 비고 |
|---|---|---|---|
| HuggingFace Endpoint | `HuggingFaceEndpointEmbeddings` | 원격(HuggingFace 서버) | 모델을 내려받지 않고 API로 호출, `HF_TOKEN` 필요 |
| HuggingFace 로컬 실행 | `HuggingFaceEmbeddings` | 로컬(CPU/GPU) | 모델 가중치를 직접 다운로드해서 실행 |
| BGE-M3 직접 사용 | `FlagEmbedding.BGEM3FlagModel` | 로컬 | LangChain 래퍼 없이 원 라이브러리로 세 가지 임베딩 방식 모두 활용 |

`BAAI/bge-m3`는 한 모델에서 세 가지 임베딩을 동시에 지원하는 것이 특징이다.

- **dense**: 문장 전체를 하나의 벡터로 압축 — 의미적 유사도 비교에 사용.
- **sparse(lexical)**: 문장에 등장하는 단어별 가중치 — 키워드 매칭에 가까움, 같은 단어를 공유하지 않으면 점수 0.
- **ColBERT(multi-vector)**: 토큰마다 벡터를 가져 더 세밀한 매칭 — 같은 문장은 1.0, 다른 문장은 그보다 낮은 점수.

실무에서는 이 세 방식을 조합(하이브리드 검색)해서 의미 기반 검색과 키워드 기반 검색의 장점을 모두 활용하기도 한다.